# EED Article 26 Compliance Analysis: Three Facility Comparison

This notebook demonstrates how deterministic compliance metrics reveal the difference between **hardware claims** and **auditable performance** across three hypothetical data-center facilities.

All calculations use the `eed-compliance-engine` core with zero LLM inference, zero stochastic sampling, and full input provenance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from decimal import Decimal
from datetime import datetime

from core.validators import (
    TelemetryBundle, ReportingPeriod, EnergyMeters,
    CoolingTelemetry, WaterUsage, CarbonIntensity
)
from core.metrics import generate_compliance_metrics
from audit.logger import log_calculation

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Sample Telemetry

Three facilities with different operational profiles:

| Facility | IT Load | Total Energy | Reused Energy | Renewable Fraction | Cooling Delta |
|----------|---------|--------------|---------------|-------------------|---------------|
| **AMS-HVN-01** (Base) | 10 GWh | 11 GWh | 1.5 GWh | 40% | 15K |
| **AMS-HVN-02** (High Renewables) | 5 GWh | 6 GWh | 0.8 GWh | 60% | 15K |
| **AMS-HVN-01 H1** (Half-Year) | 4.5 GWh | 5 GWh | 0.5 GWh | 30% | 16K |

The key question: which facility presents the strongest compliance story to a grid authority or investor?

In [ ]:
df = pd.read_csv('data/sample_telemetry.csv')
df

## 2. Construct Telemetry Bundles

Each row becomes a validated `TelemetryBundle` before any calculation runs. Pydantic enforces physical constraints (IT <= Total, outlet > inlet, dates chronological) at the boundary.

In [ ]:
def row_to_bundle(row):
    return TelemetryBundle(
        period=ReportingPeriod(
            start=datetime.strptime(row['period_start'], '%Y-%m-%d'),
            end=datetime.strptime(row['period_end'], '%Y-%m-%d'),
            reporting_entity=row['reporting_entity'],
            facility_id=row['facility_id'],
        ),
        energy=EnergyMeters(
            it_equipment_energy_kwh=row['it_energy_kwh'],
            total_facility_energy_kwh=row['total_energy_kwh'],
            reused_energy_kwh=row['reused_energy_kwh'],
        ),
        cooling=CoolingTelemetry(
            inlet_temp_c=row['cooling_inlet_c'],
            outlet_temp_c=row['cooling_outlet_c'],
            flow_rate_lph=row['flow_rate_lph'],
        ),
        water=WaterUsage(annual_water_usage_litres=row['annual_water_litres']),
        carbon=CarbonIntensity(
            grid_carbon_intensity_g_co2_kwh=row['grid_ci_g_co2_kwh'],
            on_site_renewable_fraction=row['renewable_fraction'],
        ),
    )

bundles = [row_to_bundle(row) for _, row in df.iterrows()]
for b in bundles:
    print(f"{b.period.facility_id} | {b.period.start.date()} -> {b.period.end.date()} | validated")

## 3. Compute Compliance Metrics

All metrics are deterministic pure functions. The same inputs always produce the same outputs, to 3–4 decimal places, using `Decimal` arithmetic to avoid IEEE 754 drift.

In [ ]:
results = []
record_ids = []

for bundle in bundles:
    metrics = generate_compliance_metrics(bundle)
    record_id = log_calculation(bundle, metrics, caller='analysis-notebook')
    record_ids.append(record_id)
    results.append({
        'facility_id': bundle.period.facility_id,
        'period': f"{bundle.period.start.date()} to {bundle.period.end.date()}",
        'pue': float(metrics['pue']),
        'erf': float(metrics['erf']),
        'ere': float(metrics['ere']),
        'wue': float(metrics['wue']),
        'cue': float(metrics['cue']),
        'recovered_kw': float(metrics['recovered_thermal_power_kw']),
        'recovered_kwh': float(metrics['annual_recovered_energy_kwh']),
        'renewable_frac': bundle.carbon.on_site_renewable_fraction,
    })

results_df = pd.DataFrame(results)
results_df

## 4. PUE vs ERF: The Compliance Trade-off

PUE measures electrical efficiency. ERF measures how much energy is reused outside the boundary.

**The insight:** A facility can have a good PUE (low electrical overhead) but a poor ERF (wasting recoverable heat). Investors and grid authorities increasingly look at **both** — and the German EnEfG mandates minimum ERF thresholds (10% from July 2026, 20% from 2028).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#c2551b', '#1f6570', '#d59a3a']
labels = ['AMS-HVN-01 (Base)', 'AMS-HVN-02 (High Renewables)', 'AMS-HVN-01 H1']

for i, row in results_df.iterrows():
    ax.scatter(row['pue'], row['erf'], s=300, c=colors[i], edgecolors='white', linewidth=2, zorder=5)
    ax.annotate(labels[i], (row['pue'], row['erf']),
                textcoords='offset points', xytext=(12, 0), fontsize=10)

ax.axhline(y=0.10, color='red', linestyle='--', alpha=0.5, label='Germany EnEfG 2026 (10%)')
ax.axhline(y=0.20, color='darkred', linestyle='--', alpha=0.5, label='Germany EnEfG 2028 (20%)')
ax.set_xlabel('PUE (lower is better)', fontsize=11)
ax.set_ylabel('ERF (higher is better)', fontsize=11)
ax.set_title('PUE vs Energy Reuse Factor: Three Facility Profiles', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(1.05, 1.25)
ax.set_ylim(0, 0.25)
plt.tight_layout()
plt.show()

## 5. ERE: The Combined Efficiency Score

ERE = (1 − ERF) × PUE

**Interpretation:**
- ERE < PUE means heat reuse is improving overall efficiency
- The larger the gap between PUE and ERE, the more valuable the heat recovery system is

Which facility turns waste heat into the biggest efficiency gain?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = range(len(results_df))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], results_df['pue'], width, label='PUE', color='#5c6670')
bars2 = ax.bar([i + width/2 for i in x], results_df['ere'], width, label='ERE', color='#1f6570')

ax.set_ylabel('Efficiency Ratio (lower is better)')
ax.set_title('PUE vs ERE: Heat Reuse Impact on Effective Efficiency')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0.8, 1.15)

# Add gap annotations
for i, row in results_df.iterrows():
    gap = row['pue'] - row['ere']
    ax.annotate(f"gap: {gap:.4f}",
                xy=(i, row['ere'] + gap/2),
                ha='center', fontsize=9, color='#c2551b', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nEfficiency gain from heat reuse (PUE - ERE):\n")
for i, row in results_df.iterrows():
    gap = row['pue'] - row['ere']
    print(f"{row['facility_id']}: {gap:.4f} ({gap/row['pue']*100:.1f}% improvement)")

## 6. Carbon Intensity: Grid vs On-Site Renewables

CUE shows the carbon cost per unit of IT work. Two drivers:
1. Grid carbon intensity (g CO₂/kWh) — where you are
2. On-site renewable fraction — what you generate yourself

**AMS-HVN-02** has the lowest CUE not because it uses less power, but because 60% of its energy comes from on-site renewables.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.barh(labels, results_df['cue'], color=['#c2551b', '#1f6570', '#d59a3a'])
ax.set_xlabel('CUE (kg CO₂ / kWh IT)')
ax.set_title('Carbon Usage Effectiveness by Facility')

for i, (bar, row) in enumerate(zip(bars, results_df.itertuples())):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{row.cue:.3f}\n({row.renewable_frac*100:.0f}% renewable)",
            va='center', fontsize=9)

ax.set_xlim(0, max(results_df['cue']) * 1.3)
plt.tight_layout()
plt.show()

## 7. Recovered Thermal Power: From Physics to Revenue

Q = ṁ × cp × ΔT

The liquid cooling loop in AMS-HVN-01 is recovering enough thermal power to feed a meaningful district heating offtake. The question for investors is not *can the hardware do it?* but *is there a signed agreement with a district heating operator?*

Hardware without offtake = stranded asset.
Hardware + signed offtake = grid-fit asset.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(labels, results_df['recovered_kw'], color=['#c2551b', '#1f6570', '#d59a3a'])
ax.set_ylabel('Recovered Thermal Power (kW)')
ax.set_title('Waste Heat Recovery Potential from Liquid Cooling Loop')

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.0f} kW',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nAnnual recovered energy potential:\n")
for i, row in results_df.iterrows():
    print(f"{row['facility_id']}: {row['recovered_kwh']:,.0f} kWh/year")

## 8. Key Insight: What the Metrics Reveal

### AMS-HVN-01 (Base)
- **PUE 1.100** — Efficient electrical infrastructure
- **ERF 0.1364** — Above the 2026 German threshold (10%), below the 2028 threshold (20%)
- **Recovered 871 kW thermal** — Significant district heating potential
- **Risk:** Without a signed offtake agreement, this heat is not monetized and does not count as 'maturity' in queue re-ranking

### AMS-HVN-02 (High Renewables)
- **PUE 1.200** — Less efficient electrically than HVN-01
- **ERF 0.1333** — Similar heat reuse to HVN-01
- **ERE 1.0400** — Heat reuse improves effective efficiency, but the gap is smaller
- **CUE 0.0720** — **Best carbon story** due to 60% on-site renewables
- **Strategic value:** In a carbon-priced market (ETS-2 from 2027), this facility has the lowest compliance cost per unit of compute

### AMS-HVN-01 H1 (Half-Year, Lower Renewable)
- **PUE 1.111** — Slightly worse than full-year HVN-01
- **ERF 0.1000** — Exactly at the 2026 German minimum
- **CUE 0.1750** — Worst carbon story due to 30% renewable fraction and higher grid intensity
- **Risk:** Marginal compliance. Any operational degradation pushes it below threshold.

---

## Conclusion

The facility with the **best headline PUE** (AMS-HVN-01 at 1.100) is not the facility with the **strongest compliance story**.

For a 2027 European investor or grid authority evaluating 'maturity, land rights, and grid fit':

1. **AMS-HVN-02** wins on carbon-adjusted economics (lowest CUE, highest renewable share)
2. **AMS-HVN-01** wins on heat recovery volume but needs documented offtake agreements to monetize it
3. **AMS-HVN-01 H1** is operationally fragile — one bad quarter and it misses ERF compliance

**The compliance layer is not the hardware. It is the evidence that the hardware delivers what the claims say it delivers — continuously, measurably, and auditable.**

## 9. Audit Trail

Every calculation in this notebook is logged to the immutable audit trail with a SHA-256 input hash. The same inputs always produce the same outputs and the same record ID.

In [ ]:
for i, (bundle, rid) in enumerate(zip(bundles, record_ids)):
    print(f"{bundle.period.facility_id}: {rid}")

---

*Generated with `eed-compliance-engine`. All calculations are deterministic with no LLM inference. Metric formulas reference ISO/IEC 30134 series and EU Energy Efficiency Directive 2023/1791 Article 26.*